In [ ]:
# prompt: mount the drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# prompt: download the file the dock "https://humanheart-project.creatis.insa-lyon.fr/database/api/v1/collection/6373703d73e9f0047faa1bc8/download"

!wget "https://humanheart-project.creatis.insa-lyon.fr/database/api/v1/collection/6373703d73e9f0047faa1bc8/download"


--2025-02-11 10:55:11--  https://humanheart-project.creatis.insa-lyon.fr/database/api/v1/collection/6373703d73e9f0047faa1bc8/download
Resolving humanheart-project.creatis.insa-lyon.fr (humanheart-project.creatis.insa-lyon.fr)... 195.220.108.28
Connecting to humanheart-project.creatis.insa-lyon.fr (humanheart-project.creatis.insa-lyon.fr)|195.220.108.28|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/zip]
Saving to: ‘download.2’

download.2              [     <=>            ]   3.57G  1.49MB/s    in 19m 8s  

2025-02-11 11:14:19 (3.19 MB/s) - ‘download.2’ saved [3835077244]



In [ ]:
# prompt: unzip the file "/content/download.2" and store it in path "/content/drive/MyDrive"

import zipfile
import os

# Define the paths
zip_file_path = "/content/download.2"
extract_path = "/content/drive/MyDrive"

# Check if the zip file exists
if os.path.exists(zip_file_path):
  # Create the extraction directory if it doesn't exist
  os.makedirs(extract_path, exist_ok=True)

  # Extract the zip file
  try:
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
      zip_ref.extractall(extract_path)
    print(f"Successfully extracted {zip_file_path} to {extract_path}")
  except zipfile.BadZipFile:
    print(f"Error: {zip_file_path} is not a valid zip file.")
  except Exception as e:
    print(f"An error occurred: {e}")
else:
  print(f"Error: {zip_file_path} not found.")


Error: /content/download.2 not found.


In [ ]:
import os
import nibabel as nib
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
import matplotlib.pyplot as plt
import cv2
from skimage.transform import resize
from tqdm import tqdm


In [ ]:
DATASET_PATH = "/content/drive/MyDrive/CAMUS_public/database_nifti/"

In [ ]:
!pip install nibabel numpy matplotlib tensorflow keras tqdm scikit-image opencv-python
# nibabel numpy matplotlib tensorflow keras tqdm scikit-image opencv-python # Libraries used in this notebook

In [ ]:
import os
import numpy as np
import nibabel as nib
from tqdm import tqdm

DATASET_PATH = "/content/drive/MyDrive/CAMUS_public/database_nifti"
OUTPUT_PATH = "/content/drive/MyDrive/CAMUS_processed"  # New identical folder structure

def preprocess_image(file_path):
    """Loads and normalizes a NIfTI image."""
    if not os.path.exists(file_path):
        return None
    img = nib.load(file_path).get_fdata()
    img = np.squeeze(img)  # Ensure it's 2D
    img = (img - np.min(img)) / (np.max(img) - np.min(img) + 1e-8)  # Normalize
    return img.astype(np.float32)

def collect_view_data(view):
    """Collects image-ground truth pairs for ES and ED phases of a specific view (2CH or 4CH)
       and saves them into an identical new folder structure."""
    phases = ["ES", "ED"]  # Only ES and ED phases

    patients = sorted(os.listdir(DATASET_PATH))

    for patient in tqdm(patients):
        patient_path = os.path.join(DATASET_PATH, patient)
        new_patient_path = os.path.join(OUTPUT_PATH, patient)
        if not os.path.isdir(patient_path):
            continue

        os.makedirs(new_patient_path, exist_ok=True)

        try:
            for phase in phases:
                img_path = os.path.join(patient_path, f"{patient}_{view}_{phase}.nii.gz")
                seg_path = os.path.join(patient_path, f"{patient}_{view}_{phase}_gt.nii.gz")

                img = preprocess_image(img_path)
                seg = preprocess_image(seg_path)

                if img is not None and seg is not None:
                    # Create output file path inside the new patient folder
                    output_file_path = os.path.join(new_patient_path, f"{patient}_{view}_{phase}.npz")

                    # Save the image and segmentation data to a .npz file
                    np.savez_compressed(output_file_path, image=img, segmentation=seg)

        except Exception as e:
            print(f"Error processing {patient} for {view}: {e}")

    print(f"Data for {view} view saved in the new identical folder structure")

# Process 2CH and 4CH views for ES and ED phases
print("\nProcessing 2CH view...")
collect_view_data("2CH")

print("\nProcessing 4CH view...")
collect_view_data("4CH")



Processing 2CH view...


100%|██████████| 501/501 [27:45<00:00,  3.32s/it]


Data for 2CH view saved in the new identical folder structure

Processing 4CH view...


100%|██████████| 501/501 [29:56<00:00,  3.59s/it]

Data for 4CH view saved in the new identical folder structure


In [ ]:
import os
import numpy as np
import tensorflow as tf
import cv2  # OpenCV for image saving
from tqdm import tqdm  # Progress bar

# Define paths
PROCESSED_DATASET_PATH = "/content/drive/MyDrive/CAMUS_processed"
OUTPUT_PATH = "/content/drive/MyDrive/CAMPUS_GAN_INPUT"

# Create output directories
os.makedirs(os.path.join(OUTPUT_PATH, "X_train_ES"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_PATH, "Y_train_ES"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_PATH, "X_train_ED"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_PATH, "Y_train_ED"), exist_ok=True)

def load_and_save_data(phase):
    """Loads 2CH and 4CH pairs for a given phase (ES or ED), preprocesses them for GANs, and saves as images."""

    patients = sorted(os.listdir(PROCESSED_DATASET_PATH))

    for patient in tqdm(patients, desc=f"Processing {phase} data"):
        patient_path = os.path.join(PROCESSED_DATASET_PATH, patient)

        if not os.path.isdir(patient_path):
            continue

        # Define file paths
        input_file = os.path.join(patient_path, f"{patient}_2CH_{phase}.npz")
        target_file = os.path.join(patient_path, f"{patient}_4CH_{phase}.npz")

        if os.path.exists(input_file) and os.path.exists(target_file):
            try:
                # Load data
                input_data = np.load(input_file)
                target_data = np.load(target_file)

                input_image = input_data['image']
                target_image = target_data['image']

                # Normalize to [-1, 1] for GAN training
                input_image = (input_image.astype(np.float32) / 127.5) - 1
                target_image = (target_image.astype(np.float32) / 127.5) - 1

                # Convert to uint8 (scale back to [0, 255] for saving as PNG)
                input_image_save = ((input_image + 1) * 127.5).astype(np.uint8)
                target_image_save = ((target_image + 1) * 127.5).astype(np.uint8)

                # Ensure images are 2D (grayscale)
                if input_image_save.ndim == 3:
                    input_image_save = input_image_save[:, :, 0]
                if target_image_save.ndim == 3:
                    target_image_save = target_image_save[:, :, 0]

                # Save images
                cv2.imwrite(os.path.join(OUTPUT_PATH, f"X_train_{phase}", f"{patient}.png"), input_image_save)
                cv2.imwrite(os.path.join(OUTPUT_PATH, f"Y_train_{phase}", f"{patient}.png"), target_image_save)

            except Exception as e:
                print(f"Error processing {patient} for {phase}: {e}")

# Process and save both ES and ED phases
load_and_save_data("ES")
load_and_save_data("ED")

print("GAN-compatible images saved successfully.")

Processing ED data: 100%|██████████| 500/500 [06:43<00:00,  1.24it/s]

GAN-compatible images saved successfully.


In [ ]:
import os
import numpy as np
import nibabel as nib
import cv2
from tqdm import tqdm

DATASET_PATH = "/content/drive/MyDrive/CAMUS_public/database_nifti"
OUTPUT_PATH = "/content/drive/MyDrive/CAMUS_GAN_Processed"

# Use uppercase phases to match actual filenames
PHASES = ["ES", "ED"]
VIEWS = ["2CH", "4CH"]

def normalize_image(image):
    image = image.astype(np.float32)
    image = (image - np.min(image)) / (np.max(image) - np.min(image) + 1e-8)
    return (image * 2) - 1

def process_nifti_images():
    for patient_dir in tqdm(sorted(os.listdir(DATASET_PATH))):
        patient_path = os.path.join(DATASET_PATH, patient_dir)

        if not os.path.isdir(patient_path):
            continue

        # Extract patient number (assuming format "patientXXXX")
        patient_number = patient_dir.replace("patient", "")

        for phase in PHASES:
            for view in VIEWS:
                # Construct correct filename pattern
                base_name = f"patient{patient_number}_{view}_{phase}"
                original_file = os.path.join(patient_path, f"{base_name}.nii.gz")
                ground_file = os.path.join(patient_path, f"{base_name}_gt.nii.gz")

                # Create output directory
                output_dir = os.path.join(OUTPUT_PATH, phase.lower(), view, f"patient{patient_number}")
                os.makedirs(output_dir, exist_ok=True)

                # Process original image
                if os.path.exists(original_file):
                    try:
                        img = nib.load(original_file).get_fdata()
                        # Check dimensions before slicing
                        if img.ndim == 3:
                            img_slice = img[:, :, 0]  # First temporal slice if 3D
                        elif img.ndim == 2:
                            img_slice = img  # Already 2D
                        else:
                            raise ValueError(f"Unexpected number of dimensions: {img.ndim}")

                        norm_img = normalize_image(img_slice)
                        out_path = os.path.join(output_dir, f"{base_name}_original.png")
                        cv2.imwrite(out_path, ((norm_img + 1) * 127.5).astype(np.uint8))
                    except Exception as e:
                        print(f"Error with {original_file}: {str(e)}")

                # Process ground truth
                if os.path.exists(ground_file):
                    try:
                        gt = nib.load(ground_file).get_fdata()
                        if gt.ndim == 3:
                            gt_slice = gt[:, :, 0]
                        elif gt.ndim == 2:
                            gt_slice = gt
                        else:
                            raise ValueError(f"Unexpected number of dimensions: {gt.ndim}")

                        norm_gt = normalize_image(gt_slice)
                        out_path = os.path.join(output_dir, f"{base_name}_ground.png")
                        cv2.imwrite(out_path, ((norm_gt + 1) * 127.5).astype(np.uint8))
                    except Exception as e:
                        print(f"Error with {ground_file}: {str(e)}")

process_nifti_images()
print("✅ Processing complete")


100%|██████████| 501/501 [28:29<00:00,  3.41s/it]

✅ Processing complete


In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import cv2

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras import layers
import cv2

# Set image dimensions and output path
IMG_HEIGHT, IMG_WIDTH = 256, 256  # Assuming images are resized to 256x256
OUTPUT_PATH = "/content/drive/MyDrive/CAMUS_GAN_Processed"

# ---------- Generator (U-Net Based) ----------
def build_generator(input_shape=(IMG_HEIGHT, IMG_WIDTH, 2)):
    inputs = tf.keras.Input(shape=input_shape)

    # Encoder: Downsampling layers
    def downsample(x, filters, kernel_size=4, apply_batchnorm=True):
        x = layers.Conv2D(filters, kernel_size, strides=2, padding='same', use_bias=False)(x)
        if apply_batchnorm:
            x = layers.BatchNormalization()(x)
        x = layers.LeakyReLU()(x)
        return x

    # Decoder: Upsampling layers with skip connections
    def upsample(x, skip, filters, kernel_size=4, apply_dropout=False):
        x = layers.Conv2DTranspose(filters, kernel_size, strides=2, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        if apply_dropout:
            x = layers.Dropout(0.5)(x)
        x = layers.ReLU()(x)
        x = layers.Concatenate()([x, skip])
        return x

    # Downsampling path
    d1 = downsample(inputs, 64, apply_batchnorm=False)         # (128x128x64)
    d2 = downsample(d1, 128)                                     # (64x64x128)
    d3 = downsample(d2, 256)                                     # (32x32x256)
    d4 = downsample(d3, 512)                                     # (16x16x512)
    d5 = downsample(d4, 512)                                     # (8x8x512)

    # Bottleneck
    bottleneck = layers.Conv2D(512, 4, strides=2, padding='same', use_bias=False)(d5)  # (4x4x512)
    bottleneck = layers.ReLU()(bottleneck)

    # Upsampling path with skip connections
    u1 = layers.Conv2DTranspose(512, 4, strides=2, padding='same', use_bias=False)(bottleneck)
    u1 = layers.BatchNormalization()(u1)
    u1 = layers.Dropout(0.5)(u1)
    u1 = layers.ReLU()(u1)
    u1 = layers.Concatenate()([u1, d5])                           # (8x8x1024)

    u2 = upsample(u1, d4, 512, apply_dropout=True)               # (16x16x1024)
    u3 = upsample(u2, d3, 256)                                   # (32x32x512)
    u4 = upsample(u3, d2, 128)                                   # (64x64x256)
    u5 = upsample(u4, d1, 64)                                    # (128x128x128)

    u6 = layers.Conv2DTranspose(32, 4, strides=2, padding='same', use_bias=False)(u5)  # (256x256x32)
    u6 = layers.BatchNormalization()(u6)
    u6 = layers.ReLU()(u6)

    # Final layer: Output 2 channels (synthetic 4CH image & segmentation mask)
    outputs = layers.Conv2D(2, 4, strides=1, padding='same', activation='tanh')(u6)
    return tf.keras.Model(inputs, outputs, name='generator')

# ---------- Discriminator (PatchGAN) ----------
def build_discriminator(input_shape=(IMG_HEIGHT, IMG_WIDTH, 4)):
    inp = tf.keras.Input(shape=input_shape)
    x = layers.Conv2D(64, 4, strides=2, padding='same')(inp)
    x = layers.LeakyReLU()(x)
    x = layers.Conv2D(128, 4, strides=2, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)
    x = layers.Conv2D(256, 4, strides=2, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)
    x = layers.Conv2D(512, 4, strides=1, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)
    outputs = layers.Conv2D(1, 4, strides=1, padding='same')(x)
    return tf.keras.Model(inp, outputs, name='discriminator')

# Instantiate models
generator = build_generator()
discriminator = build_discriminator()

# ---------- Feature Extractor for Perceptual Loss ----------
def build_feature_extractor():
    # A simple feature extractor; replace with a domain-specific model if available.
    inp = tf.keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))
    x = layers.Conv2D(32, 3, activation='relu', padding='same')(inp)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
    return tf.keras.Model(inp, x, name='feature_extractor')

feature_extractor = build_feature_extractor()
feature_extractor.trainable = False

def perceptual_loss(y_true, y_pred, extractor):
    # Correctly select the first channel from the last axis
    y_true_image = y_true[:, :, :, 0:1]  # Now shape: (batch, 256, 256, 1)
    y_pred_image = y_pred[:, :, :, 0:1]  # Now shape: (batch, 256, 256, 1)

    # Convert the grayscale channel to three channels
    y_true_rgb = tf.image.grayscale_to_rgb(y_true_image)  # (batch, 256, 256, 3)
    y_pred_rgb = tf.image.grayscale_to_rgb(y_pred_image)  # (batch, 256, 256, 3)

    features_true = extractor(y_true_rgb)
    features_pred = extractor(y_pred_rgb)
    return tf.reduce_mean(tf.abs(features_true - features_pred))

# ---------- Loss Functions and Optimizers ----------
bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)
mae = tf.keras.losses.MeanAbsoluteError()

lambda_L1 = 100
lambda_perc = 10

def discriminator_loss(real_output, fake_output):
    real_loss = bce(tf.ones_like(real_output), real_output)
    fake_loss = bce(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss

def generator_loss(disc_fake, gen_output, target):
    gan_loss = bce(tf.ones_like(disc_fake), disc_fake)
    l1_loss = mae(target, gen_output)
    perc_loss = perceptual_loss(target, gen_output, feature_extractor)
    return gan_loss + lambda_L1 * l1_loss + lambda_perc * perc_loss

optimizer_gen = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
optimizer_disc = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)

# ---------- Data Loading Functions ----------
def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_png(image, channels=1)
    image = tf.cast(image, tf.float32)
    image = tf.image.resize(image, [IMG_HEIGHT, IMG_WIDTH])
    image = (image / 127.5) - 1  # Scale [0,255] -> [-1,1]
    return image

def load_pair(phase, patient, view_input='2CH', view_target='4CH'):
    """
    For a given phase (e.g., "ed" or "es") and patient (e.g., "0001"),
    load the 2CH images (original and ground) and 4CH images (original and ground)
    and return them as concatenated pairs.
    """
    base_name_input = f"patient{patient}_{view_input}_{phase.upper()}"
    base_name_target = f"patient{patient}_{view_target}_{phase.upper()}"

    input_dir = os.path.join(OUTPUT_PATH, phase, view_input, f"patient{patient}")
    target_dir = os.path.join(OUTPUT_PATH, phase, view_target, f"patient{patient}")

    input_original_path = os.path.join(input_dir, f"{base_name_input}_original.png")
    input_ground_path   = os.path.join(input_dir, f"{base_name_input}_ground.png")
    target_original_path = os.path.join(target_dir, f"{base_name_target}_original.png")
    target_ground_path   = os.path.join(target_dir, f"{base_name_target}_ground.png")

    input_image = load_image(input_original_path)
    input_mask  = load_image(input_ground_path)
    target_image = load_image(target_original_path)
    target_mask  = load_image(target_ground_path)

    # Concatenate to form 2-channel inputs for condition and target respectively
    condition = tf.concat([input_image, input_mask], axis=-1)  # (256,256,2)
    target    = tf.concat([target_image, target_mask], axis=-1)  # (256,256,2)
    return condition, target

def create_dataset(phase, patient_list, batch_size=8):
    conditions, targets = [], []
    for patient in patient_list:
        try:
            cond, targ = load_pair(phase, patient)
            conditions.append(cond)
            targets.append(targ)
        except Exception as e:
            print(f"Error loading patient {patient}: {e}")
    conditions = tf.stack(conditions)
    targets = tf.stack(targets)
    dataset = tf.data.Dataset.from_tensor_slices((conditions, targets))
    dataset = dataset.shuffle(buffer_size=len(conditions)).batch(batch_size)
    return dataset

# ---------- Training Parameters and Dataset ----------
EPOCHS = 300
BATCH_SIZE = 8
patient_list = [f"{i:04d}" for i in range(1, 400)]
train_dataset = create_dataset("ed", patient_list, batch_size=BATCH_SIZE)

# ---------- Training Step ----------
@tf.function
def train_step(condition, target):
    with tf.GradientTape(persistent=True) as tape:
        # Generator produces fake 4CH pair given the 2CH condition pair
        fake_target = generator(condition, training=True)

        # For the conditional discriminator: concatenate the condition with the target
        real_input = tf.concat([condition, target], axis=-1)     # (256,256,4)
        fake_input = tf.concat([condition, fake_target], axis=-1)  # (256,256,4)

        disc_real = discriminator(real_input, training=True)
        disc_fake = discriminator(fake_input, training=True)

        gen_loss = generator_loss(disc_fake, fake_target, target)
        disc_loss = discriminator_loss(disc_real, disc_fake)

    gradients_gen = tape.gradient(gen_loss, generator.trainable_variables)
    gradients_disc = tape.gradient(disc_loss, discriminator.trainable_variables)
    optimizer_gen.apply_gradients(zip(gradients_gen, generator.trainable_variables))
    optimizer_disc.apply_gradients(zip(gradients_disc, discriminator.trainable_variables))

    return gen_loss, disc_loss

# ---------- Training Loop ----------
for epoch in range(EPOCHS):
    print(f"Epoch {epoch+1}/{EPOCHS}")
    for condition, target in train_dataset:
        g_loss, d_loss = train_step(condition, target)
    print(f"  Generator Loss: {g_loss.numpy():.4f} | Discriminator Loss: {d_loss.numpy():.4f}")

# Optionally, save the models after training:
# generator.save(os.path.join(OUTPUT_PATH, "generator_ed.h5"))
# discriminator.save(os.path.join(OUTPUT_PATH, "discriminator_ed.h5"))


KeyboardInterrupt: 

In [ ]:
generator.save(os.path.join(OUTPUT_PATH, "generator_keras_ed_l1p.keras"))
discriminator.save(os.path.join(OUTPUT_PATH, "discriminator_keras_ed_l1p.keras"))

In [ ]:
import tensorflow as tf

generator = tf.keras.models.load_model("/content/drive/MyDrive/CAMUS_GAN_Processed/generator_keras_ed_l1p.keras", compile=False)
discriminator = tf.keras.models.load_model("/content/drive/MyDrive/CAMUS_GAN_Processed/discriminator_keras_ed_l1p.keras", compile=False)

In [ ]:
import os
import cv2
import tensorflow as tf
import numpy as np


# ----- UNCOMMENT THIS SECTION WHEN LOADING THE MODEL FROM PREVIOUS BLOCK ----- #

# OUTPUT_PATH = "/content/drive/MyDrive/CAMUS_GAN_Processed"
# IMG_HEIGHT , IMG_WIDTH = 256 , 256

# def load_image(path):
#     image = tf.io.read_file(path)
#     image = tf.image.decode_png(image, channels=1)
#     image = tf.cast(image, tf.float32)
#     image = tf.image.resize(image, [IMG_HEIGHT, IMG_WIDTH])
#     image = (image / 127.5) - 1  # Scale [0,255] -> [-1,1]
#     return image


# def load_pair(phase, patient, view_input='2CH', view_target='4CH'):
#     """
#     For a given phase (e.g., "ed" or "es") and patient (e.g., "0001"),
#     load the 2CH images (original and ground) and 4CH images (original and ground)
#     and return them as concatenated pairs.
#     """
#     base_name_input = f"patient{patient}_{view_input}_{phase.upper()}"
#     base_name_target = f"patient{patient}_{view_target}_{phase.upper()}"

#     input_dir = os.path.join(OUTPUT_PATH, phase, view_input, f"patient{patient}")
#     target_dir = os.path.join(OUTPUT_PATH, phase, view_target, f"patient{patient}")

#     input_original_path = os.path.join(input_dir, f"{base_name_input}_original.png")
#     input_ground_path   = os.path.join(input_dir, f"{base_name_input}_ground.png")
#     target_original_path = os.path.join(target_dir, f"{base_name_target}_original.png")
#     target_ground_path   = os.path.join(target_dir, f"{base_name_target}_ground.png")

#     input_image = load_image(input_original_path)
#     input_mask  = load_image(input_ground_path)
#     target_image = load_image(target_original_path)
#     target_mask  = load_image(target_ground_path)

#     # Concatenate to form 2-channel inputs for condition and target respectively
#     condition = tf.concat([input_image, input_mask], axis=-1)  # (256,256,2)
#     target    = tf.concat([target_image, target_mask], axis=-1)  # (256,256,2)
#     return condition, target

# --------------------------------------------------------------------- #

def generate_synthetic(patient, phase="ed"):
    """
    Given a patient ID and phase, load the 2CH pair (original and ground),
    pass it through the generator to produce a synthetic 4CH pair,
    and return the synthetic 4CH image and segmentation mask.
    """
    # Load the condition (2CH pair) for the given patient
    condition, _ = load_pair(phase, patient, view_input='2CH', view_target='4CH')
    condition = tf.expand_dims(condition, 0)  # Add batch dimension
    synthetic = generator(condition, training=False)  # Shape: (1,256,256,2)

    # Extract the channels: first = synthetic 4CH image, second = segmentation mask
    synthetic_image = synthetic[0, :, :, 0:1]
    synthetic_mask  = synthetic[0, :, :, 1:2]

    # Rescale from [-1,1] to [0,255]
    synthetic_image = ((synthetic_image + 1) * 127.5)
    synthetic_mask  = ((synthetic_mask + 1) * 127.5)

    return synthetic_image, synthetic_mask

# Example: Generate synthetic 4CH pair for patient "0001" in phase "ed"
synthetic_img, synthetic_msk = generate_synthetic("0400", phase="ed")
synthetic_img_np = synthetic_img.numpy().astype(np.uint8)
synthetic_msk_np = synthetic_msk.numpy().astype(np.uint8)

# Define output directory for synthetic results
synth_output_dir = os.path.join(OUTPUT_PATH, "ed", "4CH", "patient0400")
os.makedirs(synth_output_dir, exist_ok=True)

# Save synthetic images
cv2.imwrite(os.path.join(synth_output_dir, "patient0400_4CH_ed_synthetic_original.png"), synthetic_img_np)
cv2.imwrite(os.path.join(synth_output_dir, "patient0400_4CH_ed_synthetic_ground.png"), synthetic_msk_np)

print("✅ Inference complete, synthetic images generated and saved.")


✅ Inference complete, synthetic images generated and saved.
